# Pipeline sanity check

Repeatable checklist to run after ANY pipeline change (a decode fix, a
FEATURE_MAP edit, a re-ingest) - answers "did processing actually still
produce correct data?" without re-deriving each check by hand every time.

Each check prints PASS/FAIL/INFO and appends to a summary table at the
end. Run top to bottom; read the summary first, drill into a failing
cell's own output for detail.

Checks:
1. Row-count conservation: manifest kept_rows == sum(message_part_count) in messages.csv
2. Null rates on required canonical columns
3. text_decode_failed rate vs. known baseline
4. DCS coverage - any observed DCS value NOT in the table (relying on auto-detect)
5. rule_evaluated / rule_flagged consistency
6. record_id uniqueness (join-key integrity)
7. message_partial rate
8. timestamp range sanity

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path("..").resolve()))

from ingestion.dcs_codecs import SMPP_DCS_TABLE, SS7_DCS_TABLE, SS7_AMBIGUOUS_DCS

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

PROCESSED_DIR = Path("..") / "data" / "processed"
SOURCES = ["SMPP", "SS7"]

# checks accumulate here as (source, check_name, status, detail) - printed
# as a summary table in the last cell. status is "PASS" / "FAIL" / "INFO"
# ("INFO" = no fixed pass/fail bar, just a number worth eyeballing for
# drift against what's printed here as the known-good baseline).
results = []


def record(source, name, status, detail=""):
    results.append({"source": source, "check": name, "status": status, "detail": detail})
    print(f"[{status}] {source} - {name}" + (f": {detail}" if detail else ""))

## Load manifest + messages.csv per source

In [ ]:
manifest = pd.read_csv(PROCESSED_DIR / "ingestion_manifest.csv")

messages = {}
for source in SOURCES:
    path = PROCESSED_DIR / source / "messages.csv"
    if not path.exists():
        print(f"{source}: no messages.csv at {path} - skipping (run pipeline.py / "
              f"features.message_reassembly first)")
        continue
    df = pd.read_csv(path, low_memory=False)
    # Same "" vs NaN CSV round-trip quirk ingestion/run_ingest.py's
    # load_features_csv() documents and fixes for the per-file features
    # CSVs - applies equally here since messages.csv goes through the same
    # plain to_csv()/read_csv() round-trip. Without this, Check 2 below
    # would flag text_decode_failed rows' blank text as a false "missing
    # data" positive - it isn't missing, it's the documented empty-string
    # case read back as NaN.
    if "text" in df.columns and "text_decode_failed" in df.columns:
        df.loc[df["text_decode_failed"], "text"] = df.loc[
            df["text_decode_failed"], "text"
        ].fillna("")
    df["timestamp"] = pd.to_datetime(df["timestamp"], format="mixed")
    messages[source] = df
    print(f"{source}: {len(df)} messages loaded")

## Check 1: row-count conservation across ingestion -> reassembly

Every raw part `ingest_file()` kept must show up in exactly one reassembled
message's `message_part_count` - this is a hard conservation law (not a
heuristic): `sum(message_part_count)` across all reassembled messages must
equal the manifest's total `kept_rows` for that source. A mismatch means
reassembly dropped or double-counted parts somewhere.

In [ ]:
for source, df in messages.items():
    manifest_kept = int(manifest.loc[manifest["source"] == source, "kept_rows"].sum())
    parts_in_messages = int(df["message_part_count"].sum())
    if manifest_kept == parts_in_messages:
        record(source, "row-count conservation", "PASS",
               f"{parts_in_messages} parts accounted for")
    else:
        record(source, "row-count conservation", "FAIL",
               f"manifest kept_rows={manifest_kept} vs sum(message_part_count)={parts_in_messages}")

## Check 2: null rates on required canonical columns

`originator`/`destination`/`timestamp` should be ~never null (a real gap here means a mapping regression). `text` can be legitimately "" (text_decode_failed) but should not be NaN after `load_features_csv()`'s refill - see `ingestion/run_ingest.py`.

In [ ]:
CRITICAL_COLS = ["originator", "destination", "timestamp", "text", "record_id"]

for source, df in messages.items():
    for col in CRITICAL_COLS:
        if col not in df.columns:
            record(source, f"null-rate[{col}]", "FAIL", "column missing entirely")
            continue
        null_pct = df[col].isna().mean() * 100
        status = "FAIL" if null_pct > 0.5 else "PASS"
        record(source, f"null-rate[{col}]", status, f"{null_pct:.3f}% null")

## Check 3: text_decode_failed rate vs. known baseline

Baseline established against the full real dataset: SMPP ~0.001% (59/6,045,250 parts, all pre-existing blank/whitespace payloads - see the DCS decode verification). Flags if the rate has grown noticeably, which would suggest a decode regression.

In [ ]:
# No shared hardcoded threshold - SMPP and SS7 have genuinely different
# baseline rates (real content-quality differences, not a bug in either -
# SMPP ~0.001%, SS7 ~0.06%, both confirmed against real full-dataset scans).
# Reported as INFO with a generous per-source ceiling; only fails if a rate
# jumps far past its OWN known baseline, not the other source's.
BASELINE_MAX_PCT = {"SMPP": 0.01, "SS7": 0.5}

for source, df in messages.items():
    if "text_decode_failed" not in df.columns:
        record(source, "text_decode_failed rate", "FAIL", "column missing")
        continue
    pct = df["text_decode_failed"].mean() * 100
    ceiling = BASELINE_MAX_PCT.get(source, 0.5)
    status = "PASS" if pct <= ceiling else "FAIL"
    record(source, "text_decode_failed rate", status,
           f"{pct:.4f}% ({int(df['text_decode_failed'].sum())}/{len(df)}, ceiling={ceiling}%)")

## Check 4: DCS coverage

Any DCS value observed in real data that ISN'T in `SMPP_DCS_TABLE`/`SS7_DCS_TABLE` (or `SS7_AMBIGUOUS_DCS`) falls back to scored auto-detect (see `ingestion/dcs_codecs.py`) - not wrong, but worth knowing how often it's happening. A sudden jump means new traffic is using a DCS value the table was never built against.

In [ ]:
DCS_TABLES = {
    "SMPP": (SMPP_DCS_TABLE, {}),
    "SS7": (SS7_DCS_TABLE, SS7_AMBIGUOUS_DCS),
}

for source, df in messages.items():
    table, ambiguous = DCS_TABLES[source]
    known = set(table) | set(ambiguous)
    observed = set(df["dcs"].dropna().astype(int).unique())
    unmapped = observed - known
    unmapped_rows = df["dcs"].isin(unmapped).sum() if unmapped else 0
    status = "INFO" if unmapped_rows == 0 else "FAIL" if unmapped_rows / len(df) > 0.01 else "INFO"
    record(source, "DCS coverage", status,
           f"unmapped DCS values: {sorted(unmapped)} ({unmapped_rows} rows, "
           f"{unmapped_rows / len(df) * 100:.4f}%)")

## Check 5: rule_evaluated / rule_flagged consistency

Not a pass/fail bar - a known real-data asymmetry exists (SMPP has zero `rule_evaluated & not rule_flagged` rows, SS7 has real negatives - see `notebooks/eda_time_windows.ipynb`). This just reports current counts so a real regression (e.g. SS7 suddenly losing its negatives too) is visible at a glance.

In [ ]:
for source, df in messages.items():
    evaluated = df["rule_evaluated"] == True
    flagged = df["rule_flagged"] == True
    not_flagged = evaluated & (df["rule_flagged"] != True)
    record(source, "rule_evaluated/rule_flagged", "INFO",
           f"evaluated={int(evaluated.sum())}, flagged={int(flagged.sum())}, "
           f"evaluated-not-flagged={int(not_flagged.sum())}")

## Check 6: record_id uniqueness (join-key integrity)

Every reassembled message must have a unique `record_id` - a duplicate means two logical messages collapsed into one row somewhere in reassembly, silently losing data.

In [ ]:
for source, df in messages.items():
    n_dupes = df["record_id"].duplicated().sum()
    status = "PASS" if n_dupes == 0 else "FAIL"
    record(source, "record_id uniqueness", status, f"{n_dupes} duplicate record_id(s)")

## Check 7: message_partial rate

Baseline snapshot (not a hard pass/fail): SMPP ~0.29%, SS7 ~0.99% partial (multipart groups missing an expected part - see `features/message_reassembly.py`'s grouping-key caveat re: concat_ref collisions). Reported as INFO so drift is visible without hardcoding a brittle threshold.

In [ ]:
for source, df in messages.items():
    pct = df["message_partial"].mean() * 100
    record(source, "message_partial rate", "INFO",
           f"{pct:.3f}% ({int(df['message_partial'].sum())}/{len(df)})")

## Check 8: timestamp range sanity

Catches a parse bug (e.g. epoch 1970, or a date far in the future) that a plain null-check wouldn't - the column is non-null but the VALUE is wrong.

In [ ]:
import datetime

PLAUSIBLE_MIN = pd.Timestamp("2020-01-01")
PLAUSIBLE_MAX = pd.Timestamp.now() + pd.Timedelta(days=1)

for source, df in messages.items():
    lo, hi = df["timestamp"].min(), df["timestamp"].max()
    in_range = (lo >= PLAUSIBLE_MIN) and (hi <= PLAUSIBLE_MAX)
    status = "PASS" if in_range else "FAIL"
    record(source, "timestamp range", status, f"{lo} .. {hi}")

## Summary

In [ ]:
summary = pd.DataFrame(results)
n_fail = (summary["status"] == "FAIL").sum()

print(f"{len(summary)} checks run, {n_fail} FAIL\n")
summary